# Comparer deux distributions gaussiennes

Ce notebook est une démonstration guidée. Il réutilise le dataframe du projet
de référence ; il n'est pas nécessaire pour lancer le projet ni pour exécuter
ses tests.

Progression : **prédire → exécuter → observer → expliquer**.

Une ligne représente une mesure simulée. `value` est la valeur numérique et
`component` indique le groupe A ou B.

## Charger le projet de référence

Le notebook cherche `lab/gaussian_project_solution/` depuis la racine du cours
ou depuis le dossier `notebooks/`. Si le dossier n'est pas trouvé, le message
d'erreur indique l'organisation attendue.

In [1]:
import sys
from pathlib import Path

import pandas as pd

candidates = [
    Path.cwd() / "gaussian_solution",
    Path.cwd().parent / "gaussian_solution",
    Path.cwd() / "gaussian_solution",
]
project_root = next((path.resolve() for path in candidates if path.exists()), None)
if project_root is None:
    raise FileNotFoundError(
        "Place gaussian_solution beside the notebooks folder."
    )
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

from analysis import (
    describe_groups,
    mean_comparison,
    simulate_mean_differences,
    welch_mean_test,
)
from analysis.inference_plots import plot_sampling_variation
from distributions import DEFAULT_SEED
from distributions.gaussian_mixture import GaussianMixture
from distributions.gaussian_plots import (
    plot_box,
    plot_components,
    plot_dashboard,
    plot_ecdf,
    plot_mixture,
    plot_violin,
)

print(project_root)

J:\DEV\G404-DATA_TD\G404-Data_TD_projet_gaussian-mixture\gaussian_solution


## Retrouver exactement les données du projet

Avant d'exécuter, prédisez les effectifs de A et B. La méthode `sample()`
de `gaussian_mixture.py` génère les valeurs, conserve le dataframe dans l'objet
et renvoie ce même dataframe. Les fonctions de `gaussian_plots.py` lisent
ensuite `model.samples` et renvoient des figures sans effectuer un nouveau
tirage.

In [17]:
model = GaussianMixture(100, 18, 106, 42, weight_a=0.6)
samples = model.sample(2000, seed=DEFAULT_SEED)

print(samples.shape)
print(samples["component"].value_counts().sort_index())
samples.head()

(2000, 2)
component
A    1200
B     800
Name: count, dtype: int64


,value,component
0,105.484907,A
1,81.280286,A
2,113.508122,A
3,116.930165,A
4,64.881367,A


## Observer A et B avant de choisir un test

Comparez les effectifs, le centre et la dispersion. L'histogramme superposé
normalise séparément chaque groupe en densité : la hauteur des barres ne montre
donc pas directement la proportion 60 % / 40 %.

In [18]:
summary = describe_groups(samples)
summary.round(3)

,count,mean,median,std,q1,q3,iqr
component,,,,,,,
A,1200,99.600,100.062,17.741,87.721,111.078,23.357
B,800,101.612,102.387,43.045,73.988,130.015,56.027


In [19]:
figure = plot_components(model)
figure.show()

### Comprendre le mélange pondéré

L'histogramme gris réunit toutes les observations. Les courbes A et B sont les
**contributions théoriques pondérées** par les poids du modèle, 60 % et 40 %.
Leur somme point par point donne la densité théorique du mélange. Une
contribution pondérée n'a donc pas, seule, une aire égale à 1.

In [20]:
plot_mixture(model).show()

### Comparer position et dispersion

Les boîtes à moustaches montrent la médiane, les quartiles, la dispersion et
les valeurs atypiques de A, de B et de toutes les observations réunies. Cette
vue ne montre pas la forme détaillée de chaque distribution.

In [21]:
plot_box(model).show()

### Lire une proportion cumulée

La fonction de répartition empirique (ECDF) donne, pour chaque valeur x, la
proportion d'observations inférieures ou égales à x. À une même valeur, la
courbe la plus haute indique le groupe qui a déjà accumulé la plus grande part
de ses observations.

In [22]:
plot_ecdf(model).show()

### Lire une forme lissée sans la confondre avec un effectif

La largeur du violon est une estimation lissée de la densité locale ; elle ne
représente pas l'effectif du groupe. La boîte intérieure et la ligne moyenne
gardent des repères de position. Cette vue complète les histogrammes sans les
remplacer.

In [23]:
plot_violin(model).show()

### Réunir les quatre lectures

Le tableau de bord garde les mêmes couleurs et la même étendue numérique. Il
réunit les groupes séparés, le mélange pondéré, les violons et les
proportions cumulées. Les histogrammes viennent des observations ; les courbes
normales utilisent les paramètres connus de la simulation et ne sont pas
ajustées aux données.

In [9]:
plot_dashboard(model).show()

## Mesurer la différence observée

Dans ce support, `avg(G)` désigne la moyenne observée du groupe G. La différence
est calculée dans l'ordre **avg(B) − avg(A)**. Elle reste exprimée dans
l'unité de `value`. La différence standardisée met ce résultat en regard de la
dispersion observée, mais ne remplace pas une décision métier.

In [24]:
descriptive = mean_comparison(samples, reference="A", comparison="B")
pd.Series(descriptive)

reference                         A
comparison                        B
difference                 2.012096
standardized_difference    0.065985
dtype: object

## Moyennes égales, écarts d'échantillonnage différents

Nous construisons une situation où H₀ est vraie : A et B ont la même moyenne
génératrice 103. Nous conservons les dispersions et les effectifs du projet :
écarts-types 18 et 24, puis 1 200 observations A et 800 observations B.

Chaque répétition utilise `GaussianMixture` avec une nouvelle seed, puis calcule
**avg(B) − avg(A)**. Prédisez la forme et le centre de la
distribution de ces différences.

In [25]:
null_model = GaussianMixture(103, 18, 103, 24, weight_a=0.6)
mean_differences = simulate_mean_differences(
    null_model,
    sample_size=2000,
    repetitions=1000,
    first_seed=1404,
)
plot_sampling_variation(
    mean_differences,
    observed_difference=descriptive["difference"],
    null_model=null_model,
    sample_size=2000,
).show()

# Du descriptif à une décision formelle

Nous avons une variable quantitative, deux groupes non appariés et des
dispersions différentes. La question formelle devient : **les moyennes des
populations A et B sont-elles différentes ?**

Nous notons les modèles de population
**X₁ ∼ 𝒩(μ₁, σ₁²)** pour A et **X₂ ∼ 𝒩(μ₂, σ₂²)** pour B.

- H₀ : μ₁ = μ₂
- H₁ : μ₁ ≠ μ₂

Le test de Welch compare les moyennes de deux groupes indépendants sans imposer
des variances égales. Il ne teste pas l'égalité complète des distributions.

## Intervalle de confiance et test de Welch

L'intervalle de confiance estime les valeurs compatibles avec les observations
pour la différence de population **μ₂ − μ₁**. L'estimation observée est
**avg(B) − avg(A)**. La valeur p mesure, sous H₀ et les hypothèses
du test, la fréquence de résultats au moins aussi extrêmes.

La valeur p n'est ni la probabilité que H₀ soit vraie, ni une taille d'effet,
ni une preuve de causalité.

In [26]:
formal = welch_mean_test(
    samples,
    reference="A",
    comparison="B",
    confidence_level=0.95,
)
pd.Series(formal)

reference                      A
comparison                     B
difference              2.012096
confidence_level            0.95
confidence_low         -1.138978
confidence_high          5.16317
t_statistic             1.253066
degrees_of_freedom    981.813117
p_value                  0.21048
dtype: object

## Interpréter sans dépasser les données

Dans cette simulation, le code fixe les mécanismes : nous savons pourquoi les
moyennes diffèrent. Dans une étude réelle, un écart observé peut aussi dépendre
du recrutement, de la mesure ou d'autres changements entre les groupes.

Le test de Welch ne convient pas à des mesures avant/après sur les mêmes unités :
ce plan est apparié. Il ne répond pas non plus à une question portant sur toute
la forme des distributions.

## À vous : modifier une hypothèse

Choisissez une seule modification :

- rapprocher les deux moyennes génératrices ;
- réduire le nombre d'observations ;
- augmenter la dispersion de B ;
- simuler deux groupes issus du même mécanisme.

Puis rédigez quatre lignes :

1. **Observation** — ce qui change dans les résultats.
2. **Preuve** — les nombres ou la figure utilisés.
3. **Limite** — ce que cette analyse n'établit pas.
4. **Vérification suivante** — l'information ou le calcul à obtenir.